# EPO MinerU RAG with Reranker

In [1]:
# =========================
# 1. Install dependencies
# =========================
# !pip install -q \
#   langchain \
#   langchain-community \
#   langchain-huggingface \
#   sentence-transformers \
#   faiss-cpu \
#   rank_bm25 \
#   transformers \
#   accelerate \
#   openai

# Install MinerU once if the ingest cell says the CLI is missing.
# MinerU requires Python 3.10-3.13.
# !pip install -q "mineru[all]"

# =========================
# 2. Shared imports
# =========================
from pathlib import Path
from itertools import islice
from typing import Any, Dict, List, Optional, Sequence
import hashlib
import json
import os
import shutil

import torch
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder
from openai import OpenAI

print("Setup ready.")


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup ready.


In [2]:
from ingest import INGEST
MODE = "one"  # Change to "all" to process every PDF in PDF_DIR.
ONE_FILE_PAGE_RANGE = None  # Example for a quick test: (0, 2). Use None for the full first PDF.

PDF_DIR = "pdf_files_epo"
OUTPUT_DIR = "ingested_data_epo"
PDF_LANG = "en"

In [3]:
# # =========================
# # RUN: MinerU ingest playground
# # =========================
# # MODE = "one"  -> quick test on the first PDF
# # MODE = "all"  -> parse + chunk every PDF inside PDF_DIR

# from ingest import INGEST

# MODE = "one"  # Change to "all" to process every PDF in PDF_DIR.
# ONE_FILE_PAGE_RANGE = None  # Example for a quick test: (0, 2). Use None for the full first PDF.

# PDF_DIR = "pdf_files_epo"
# OUTPUT_DIR = "ingested_data_epo"
# PDF_LANG = "en"

# # More stable on large PDFs on macOS/CPU: smaller MinerU batches and fewer threads.
# MINERU_ENV = {
#     "MINERU_PROCESSING_WINDOW_SIZE": "8",
#     "MINERU_PDF_RENDER_THREADS": "1",
#     "MINERU_API_MAX_CONCURRENT_REQUESTS": "1",
#     "MINERU_INTRA_OP_NUM_THREADS": "1",
#     "MINERU_INTER_OP_NUM_THREADS": "1",
# }

# extra_mineru_args = []
# if MODE == "one" and ONE_FILE_PAGE_RANGE is not None:
#     start_page, end_page = ONE_FILE_PAGE_RANGE
#     extra_mineru_args.extend(["-s", str(start_page), "-e", str(end_page)])

# if shutil.which("mineru") is None:
#     raise RuntimeError(
#         "MinerU CLI was not found. Install it in a Python 3.10-3.13 environment first: "
#         "uv pip install -U 'mineru[all]'"
#     )

# ingest = INGEST(
#     pdf_dir=PDF_DIR,
#     output_dir=OUTPUT_DIR,
#     backend="pipeline",
#     method="auto",
#     lang=PDF_LANG,
#     table_row_threshold=25,
#     table_char_threshold=4000,
#     skip_existing_mineru=(MODE == "all"),
#     extra_mineru_args=extra_mineru_args,
#     mineru_env=MINERU_ENV,
# )

# if MODE == "one":
#     ingest._prepare_output_dirs()
#     pdfs = sorted(Path(PDF_DIR).glob("**/*.pdf"))
#     if not pdfs:
#         raise FileNotFoundError(f"No PDFs found in {PDF_DIR}")

#     result, chunks = ingest.ingest_pdf(pdfs[0])
#     ingest._write_jsonl(ingest.all_chunks_path, chunks)
#     ingest._write_manifest([result])

#     print("Parsed one PDF:", result.source_pdf)
#     print("Page range:", ONE_FILE_PAGE_RANGE)
#     print("Markdown:", result.markdown_path)
#     print("Chunks:", result.chunks_path, "| count:", result.chunk_count)
#     print("All chunks:", ingest.all_chunks_path)
#     print("Tables dir:", result.tables_dir, "| tables:", result.table_count)
#     print("Images dir:", result.images_dir, "| images:", result.image_count)

#     print("\nFirst chunks preview:")
#     for chunk in chunks[:3]:
#         preview = chunk["text"][:500].replace("\n", " ")
#         print("-", chunk["id"], chunk["metadata"].get("chunk_type"), preview)

# elif MODE == "all":
#     results = ingest.run()
#     print("Parsed PDFs:", len(results))
#     print("All chunks:", ingest.all_chunks_path)
#     print("Manifest:", ingest.manifest_path)
#     for result in results:
#         print("-", Path(result.source_pdf).name, "chunks:", result.chunk_count, "tables:", result.table_count)

#     print("\nFirst lines from all_chunks.jsonl:")
#     with open(ingest.all_chunks_path, "r", encoding="utf-8") as f:
#         for line in islice(f, 3):
#             chunk = json.loads(line)
#             preview = chunk["text"][:500].replace("\n", " ")
#             print("-", chunk["id"], chunk["metadata"].get("chunk_type"), preview)

# else:
#     raise ValueError('MODE must be "one" or "all"')


In [4]:
# =========================
# Build / load embeddings + BM25 from the ingest folder
# =========================
INGESTED_DATA_DIR = Path(globals().get("OUTPUT_DIR", "ingested_data_epo"))
INDEX_NAME = INGESTED_DATA_DIR.name
VECTOR_STORE_DIR = Path("vector_stores")
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
REBUILD_VECTOR_STORE = False
BM25_CANDIDATES = 80
VECTOR_CANDIDATES = 80

vector_store_path = VECTOR_STORE_DIR / f"{INDEX_NAME}_faiss_reranker"
index_meta_path = vector_store_path / "index_meta.json"


def discover_chunk_paths(ingested_dir: Path) -> List[Path]:
    chunks_dir = ingested_dir / "chunks"
    all_chunks = chunks_dir / "all_chunks.jsonl"
    if all_chunks.exists():
        return [all_chunks]
    paths = sorted(path for path in chunks_dir.glob("*.jsonl") if path.name != "all_chunks.jsonl")
    if not paths:
        raise FileNotFoundError(f"No chunk JSONL files found under {chunks_dir}")
    return paths


def fingerprint_files(paths: Sequence[Path]) -> str:
    digest = hashlib.sha1()
    for path in paths:
        digest.update(path.as_posix().encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()


def load_documents_from_chunks(ingested_dir: Path) -> tuple[List[Document], List[Path], str]:
    chunk_paths = discover_chunk_paths(ingested_dir)
    fingerprint = fingerprint_files(chunk_paths)
    documents: List[Document] = []

    for chunk_path in chunk_paths:
        with chunk_path.open("r", encoding="utf-8") as file:
            for line in file:
                if not line.strip():
                    continue
                chunk = json.loads(line)
                text = (chunk.get("text") or chunk.get("content") or "").strip()
                if not text:
                    continue
                metadata = dict(chunk.get("metadata") or {})
                metadata.update(
                    {
                        "chunk_id": chunk.get("id"),
                        "chunk_file": str(chunk_path),
                        "index_name": INDEX_NAME,
                    }
                )
                documents.append(Document(page_content=text, metadata=metadata))

    if not documents:
        raise ValueError(f"Chunk files were found under {ingested_dir}, but no text chunks were loaded.")
    return documents, chunk_paths, fingerprint


def load_index_meta(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


documents, chunk_paths, chunk_fingerprint = load_documents_from_chunks(INGESTED_DATA_DIR)
current_index_meta = {
    "index_name": INDEX_NAME,
    "ingested_data_dir": str(INGESTED_DATA_DIR),
    "chunk_paths": [str(path) for path in chunk_paths],
    "chunk_count": len(documents),
    "chunk_fingerprint": chunk_fingerprint,
    "embedding_model": EMBEDDING_MODEL,
}

print(f"Corpus: {INDEX_NAME}")
print(f"Loaded chunks: {len(documents):,}")
print(f"Chunk files: {[str(path) for path in chunk_paths]}")
print(f"Loading embeddings: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

existing_index_meta = load_index_meta(index_meta_path)
index_is_current = (
    vector_store_path.exists()
    and existing_index_meta.get("chunk_fingerprint") == chunk_fingerprint
    and existing_index_meta.get("embedding_model") == EMBEDDING_MODEL
)

if index_is_current and not REBUILD_VECTOR_STORE:
    print(f"Loading existing FAISS index: {vector_store_path}")
    vector_store = FAISS.load_local(
        str(vector_store_path),
        embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    print(f"Building FAISS index: {vector_store_path}")
    vector_store = FAISS.from_documents(documents=documents, embedding=embeddings)
    vector_store.save_local(str(vector_store_path))
    index_meta_path.write_text(json.dumps(current_index_meta, indent=2), encoding="utf-8")

print("Building BM25 retriever from the same chunks.")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = BM25_CANDIDATES

print(f"Loading reranker: {RERANKER_MODEL}")
reranker_device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    cross_encoder = CrossEncoder(
        RERANKER_MODEL,
        model_kwargs={"torch_dtype": "auto"},
        trust_remote_code=True,
        device=reranker_device,
    )
except TypeError:
    cross_encoder = CrossEncoder(
        RERANKER_MODEL,
        automodel_args={"torch_dtype": "auto"},
        trust_remote_code=True,
        device=reranker_device,
    )


def _doc_key(doc: Document) -> str:
    return str(doc.metadata.get("chunk_id") or hashlib.sha1(doc.page_content.encode("utf-8")).hexdigest())


def retrieve_and_rerank(
    query: str,
    top_k: int = 14,
    candidates_k: int = 80,
    alpha: float = 0.55,
) -> List[Document]:
    candidates_k = min(candidates_k, len(documents))
    if candidates_k <= 0:
        return []

    vector_docs = vector_store.similarity_search(query, k=candidates_k)
    bm25_retriever.k = candidates_k
    bm25_docs = bm25_retriever.invoke(query)

    fused: Dict[str, Dict[str, Any]] = {}

    def add_score(found_docs: Sequence[Document], weight: float) -> None:
        for rank, doc in enumerate(found_docs):
            key = _doc_key(doc)
            if key not in fused:
                fused[key] = {"doc": doc, "score": 0.0}
            fused[key]["score"] += weight * (1.0 / (rank + 60))

    add_score(vector_docs, alpha)
    add_score(bm25_docs, 1 - alpha)

    candidates = [item["doc"] for item in sorted(fused.values(), key=lambda item: item["score"], reverse=True)]
    candidates = candidates[:candidates_k]
    if not candidates:
        return []

    pairs = [[query, doc.page_content] for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for doc, score in zip(candidates, scores):
        doc.metadata["rerank_score"] = float(score)

    return sorted(candidates, key=lambda doc: doc.metadata.get("rerank_score", 0.0), reverse=True)[:top_k]

print("Retrieval indexes are ready.")
print(f"FAISS path: {vector_store_path}")


Corpus: ingested_data_epo
Loaded chunks: 2,603
Chunk files: ['ingested_data_epo/chunks/ISG_en_4web.jsonl']
Loading embeddings: sentence-transformers/all-mpnet-base-v2
Loading existing FAISS index: vector_stores/ingested_data_epo_faiss_reranker
Building BM25 retriever from the same chunks.
Loading reranker: BAAI/bge-reranker-v2-m3


`torch_dtype` is deprecated! Use `dtype` instead!


Retrieval indexes are ready.
FAISS path: vector_stores/ingested_data_epo_faiss_reranker


In [ ]:
# =========================
# EPO / Interinstitutional Style Guide Q&A
# =========================
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")
client = OpenAI(api_key=OPENAI_API_KEY or None)

ANSWER_SYSTEM_PROMPT = """You are a careful retrieval-grounded assistant for the EPO corpus in this notebook: the Publications Office of the European Union / Interinstitutional Style Guide material.

Your job is to answer the user's actual question using the retrieved context first. Be practical, precise, and helpful. Use exact rules, names, numbers, examples, and exceptions when the context supports them. Do not invent details. If the context supports only a partial answer, give the useful part and clearly say what is not covered.

Write in the user's language unless the user asks for another language. Use concise headings or bullets when they make the answer easier to read. Do not over-explain the retrieval process."""

CONTEXT_CHECK_SYSTEM_PROMPT = """You judge whether retrieved passages are sufficient to answer a user's question about the Publications Office / Interinstitutional Style Guide corpus.

Be pragmatic: sufficient means the assistant can give a useful, responsible answer, not that every possible detail has been retrieved. Mark the context insufficient only when a missing rule, definition, table, exception, or example would materially change the answer.

If more retrieval is needed, produce one targeted search query for the missing information. The new_query must be a retrieval query, not a request for the user to clarify. Avoid broad restatements of the original question.

Return only JSON with keys: sufficient, reason, new_query."""


def generate_answer_with_gpt(prompt: str) -> str:
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
        )
        return response.choices[0].message.content
    except Exception as exc:
        return f"OpenAI request failed: {exc}"


def check_if_context_sufficient(query: str, context: str) -> Dict[str, Any]:
    check_prompt = f"""Question:
{query}

Retrieved context:
{context}

Decide whether the context is enough for a useful answer. If it is not enough, write one focused retrieval query that could fetch the missing rule, section, table, example, or exception.

Return JSON only:
{{
  "sufficient": true,
  "reason": "short explanation",
  "new_query": ""
}}"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": CONTEXT_CHECK_SYSTEM_PROMPT},
                {"role": "user", "content": check_prompt},
            ],
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)
    except Exception as exc:
        print(f"Context check failed, continuing with the retrieved context: {exc}")
        return {"sufficient": True, "reason": "Context check failed; using retrieved context.", "new_query": ""}


def source_label(doc: Document, index: int) -> str:
    meta = doc.metadata
    source = meta.get("source_pdf") or meta.get("source_pdf_path") or meta.get("chunk_file") or "source"
    chunk_id = meta.get("chunk_id", f"chunk-{index}")
    chunk_type = meta.get("chunk_type", "text")
    page_start = meta.get("page_start")
    page_end = meta.get("page_end")
    if page_start and page_end and page_start != page_end:
        pages = f"pages {page_start}-{page_end}"
    elif page_start:
        pages = f"page {page_start}"
    else:
        pages = "page unknown"
    return f"Source {index} | {chunk_id} | {source} | {pages} | {chunk_type}"


def build_context(docs: Sequence[Document], max_chars: int = 60000) -> str:
    sections: List[str] = []
    used_chars = 0
    for index, doc in enumerate(docs, start=1):
        header = source_label(doc, index)
        text = doc.page_content.strip()
        remaining = max_chars - used_chars - len(header) - 8
        if remaining <= 0:
            break
        if len(text) > remaining:
            text = text[:remaining].rstrip() + "..."
        section = f"[{header}]\n{text}"
        sections.append(section)
        used_chars += len(section)
    return "\n\n".join(sections)


def unique_docs(existing: Sequence[Document], new_docs: Sequence[Document]) -> List[Document]:
    output: List[Document] = []
    seen = set()
    for doc in list(existing) + list(new_docs):
        key = doc.metadata.get("chunk_id") or doc.page_content
        if key in seen:
            continue
        output.append(doc)
        seen.add(key)
    return output


def rag_answer(
    query: str,
    max_iterations: int = 3,
    top_k: int = 14,
    candidates_k: int = 80,
    context_char_limit: int = 60000,
):
    all_docs: List[Document] = []
    current_query = query
    context = ""

    print("\n" + "=" * 60)
    print(f"Question: {query}")
    print("=" * 60)

    for iteration in range(max_iterations):
        print(f"Iteration {iteration + 1}/{max_iterations}")
        print(f"Search query: {current_query}")

        docs = retrieve_and_rerank(current_query, top_k=top_k, candidates_k=candidates_k)
        all_docs = unique_docs(all_docs, docs)
        context = build_context(all_docs, max_chars=context_char_limit)
        print(f"Retrieved unique chunks: {len(all_docs)} | context chars: {len(context):,}")

        check_result = check_if_context_sufficient(query, context)
        print(f"Context check: {check_result.get('reason', 'No reason returned')}")

        if check_result.get("sufficient", False):
            print("Context is sufficient. Writing the answer.\n")
            break

        new_query = (check_result.get("new_query") or "").strip()
        if not new_query or new_query.lower() == current_query.lower():
            print("No useful follow-up retrieval query was produced. Writing with available context.\n")
            break
        current_query = new_query
    else:
        print("Reached the iteration limit. Writing with available context.\n")

    final_prompt = f"""Use the retrieved context below to answer the user's question.

Retrieved context:
{context}

User question:
{query}

Answering requirements:
- Answer the question directly first.
- Ground the answer in the retrieved context.
- Mention specific rules, section names, examples, numbers, or exceptions when available.
- If the context is incomplete, say exactly what is missing and still provide the best supported answer.
- Keep the answer polished and easy to scan.
"""

    answer = generate_answer_with_gpt(final_prompt)
    return answer, all_docs


print("EPO RAG pipeline ready.")
print(f"Model: {LLM_MODEL}")
print(f"Corpus: {INDEX_NAME} | chunks: {len(documents):,}")
print("Ask about the Interinstitutional Style Guide, drafting rules, references, tables, abbreviations, style, and related Publications Office guidance.")

while True:
    query = input("\nAsk a question (or 'exit'): ").strip()
    if query.lower() in {"exit", "quit"}:
        print("Done.")
        break
    if not query:
        continue

    answer, sources = rag_answer(query, max_iterations=3)

    print("\n" + "=" * 70)
    print("FINAL ANSWER")
    print("=" * 70 + "\n")
    print(answer)

    print("\n" + "=" * 70)
    print(f"SOURCES ({len(sources)} reranked chunks)")
    print("=" * 70)

    for index, doc in enumerate(sources[:8], start=1):
        score = doc.metadata.get("rerank_score")
        score_text = f" | score: {score:.4f}" if isinstance(score, float) else ""
        print(f"\n{index}. {source_label(doc, index)}{score_text}")
        print("-" * 70)
        preview = doc.page_content[:350].replace("\n", " ")
        print(f"{preview}...")

    if len(sources) > 8:
        print(f"\n... plus {len(sources) - 8} more chunks")


EPO RAG pipeline ready.
Model: gpt-5.5
Corpus: ingested_data_epo | chunks: 2,603
Ask about the Interinstitutional Style Guide, drafting rules, references, tables, abbreviations, style, and related Publications Office guidance.

Question: How do citations in the preamble connect a legal act to both primary law and secondary law?
Iteration 1/3
Search query: How do citations in the preamble connect a legal act to both primary law and secondary law?
Retrieved unique chunks: 14 | context chars: 21,039
Context check: The retrieved passages explain that the preamble contains citations, that citations state the legal basis of the act, and distinguish how primary law and secondary law are cited: primary acts such as Treaties are cited as the general basis without footnotes, while secondary acts may be cited as the specific basis with full title and Official Journal footnote.
Context is sufficient. Writing the answer.


FINAL ANSWER

Citations in the preamble connect a legal act to its legal bas

In [1]:
import sys

sys.executable

'/opt/homebrew/opt/python@3.10/bin/python3.10'